# 🗂️ Paris Sales EDA — Full Deep Analysis
### Data sources
| File | Description |
|---|---|
| `FR_Outlets_Extract_Paris.xlsx` | Store master data (958 outlets) |
| `FR_SKU_Extract.xlsx` | Product catalogue (4 134 SKUs) |
| `Logista_Paris_Jan_Sell_In.xlsx` | Distributor sell-in, January 2026 |
| `Bimedia_Sell_Out_Jan.xlsx` | POS sell-out, January 2026 |

**Change `DATA_PATH` in the Setup cell to point to your local folder.**


---
## 0 · Setup — imports, paths & global style

In [ ]:


import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, kurtosis, skew
from scipy.signal import correlate

# ── ⚙️  CHANGE THIS to the folder containing your four Excel files ───────────
DATA_PATH = "."          # e.g.  r"C:/Users/you/data"
# ─────────────────────────────────────────────────────────────────────────────

# Colour palette
BLUE, CORAL, TEAL, PURPLE = "#378ADD", "#D85A30", "#1D9E75", "#7F77DD"
AMBER, RED, GRAY, GREEN   = "#BA7517", "#E24B4A", "#888780", "#639922"
PALETTE = [BLUE, CORAL, TEAL, PURPLE, AMBER, RED, GRAY, GREEN]

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False,    "axes.spines.right": False,
    "axes.grid": True,           "grid.color": "#e8e8e8",
    "grid.linewidth": 0.6,       "font.size": 10,
    "axes.titlesize": 12,        "axes.titleweight": "bold",
    "figure.dpi": 110,
})

def fmt_k(x, _=None):
    if abs(x) >= 1_000_000: return f"{x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"{x/1_000:.0f}K"
    return str(int(x))

print("✅  Setup complete")


---
## 1 · Load & Clean Data

In [ ]:
# ── 1.1  Load raw files ──────────────────────────────────────────────────────
outlets  = pd.read_excel(os.path.join(DATA_PATH, "FR_Outlets_Extract_Paris.xlsx"))
skus     = pd.read_excel(os.path.join(DATA_PATH, "FR_SKU_Extract.xlsx"))
sell_in  = pd.read_excel(os.path.join(DATA_PATH, "Logista_Paris_Jan_Sell_In.xlsx"))
sell_out = pd.read_excel(os.path.join(DATA_PATH, "Bimedia_Sell_Out_Jan.xlsx"))

print(f"outlets  : {outlets.shape}")
print(f"skus     : {skus.shape}")
print(f"sell_in  : {sell_in.shape}")
print(f"sell_out : {sell_out.shape}")


In [ ]:
# ── 1.2  Rename columns for convenience ─────────────────────────────────────
outlets.columns = [
    "store_code","outlet_name","outlet_sf","status","city","state",
    "latitude","longitude","ownership_type","territory_id","terr_id2",
    "terr_name","area","em_region","division"
]

skus.columns = [
    "shortcode","record_type","sku_name","sku_code","bv_code",
    "brand_variant","bf_code","brand_family","category","sku_status"
]

sell_in.columns = [
    "sales_date","pos_code","outlet_sf","sku_code","sku_sf",
    "sku_record","delivery_date","base_unit","sales_unit"
]

sell_out.columns = [
    "sales_date","pos","outlet_sf","city","sku",
    "sku_sf","record_type","vol_unit","vol_packs"
]

print("✅  Columns renamed")


In [ ]:
# ── 1.3  Parse dates & derive time features ──────────────────────────────────
DOW_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

for df, date_cols in [
    (sell_in,  ["sales_date","delivery_date"]),
    (sell_out, ["sales_date"])
]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col].astype(str).str[:8], format="%Y%m%d", errors="coerce")

for df in [sell_in, sell_out]:
    df["dow"]  = df["sales_date"].dt.day_name()
    df["week"] = df["sales_date"].dt.isocalendar().week.astype(int)
    df["day"]  = df["sales_date"].dt.day

sell_in["lead_days"]    = (sell_in["delivery_date"] - sell_in["sales_date"]).dt.days
sell_out["is_weekend"]  = sell_out["sales_date"].dt.dayofweek >= 5

print("✅  Dates parsed")


In [ ]:
# ── 1.4  Filtered sets (exclude SKU 99999 placeholder) ───────────────────────
sell_out_f = sell_out[sell_out["sku"]      != 99999].copy()
sell_in_f  = sell_in[sell_in["sku_code"]  != 99999].copy()

# ── 1.5  Trade SKU lookup & enrichment ──────────────────────────────────────
trade_skus = skus[skus["record_type"] == "Trade SKU"].copy()
sku_map    = trade_skus.set_index("sku_code")[
    ["brand_family","category","sku_status","sku_name"]
].to_dict("index")

sell_out_f["category"]     = sell_out_f["sku_sf"].map(lambda x: sku_map.get(x,{}).get("category","Unknown"))
sell_out_f["brand_family"] = sell_out_f["sku_sf"].map(lambda x: sku_map.get(x,{}).get("brand_family","Unknown"))
sell_out_f["sku_status"]   = sell_out_f["sku_sf"].map(lambda x: sku_map.get(x,{}).get("sku_status","Unknown"))

# ── 1.6  Handy daily aggregates ──────────────────────────────────────────────
daily_raw = sell_out.groupby("sales_date")["vol_unit"].sum()
daily_fil = sell_out_f.groupby("sales_date")["vol_unit"].sum()

print(f"sell_out_f : {sell_out_f.shape}  (excl. SKU 99999)")
print(f"sell_in_f  : {sell_in_f.shape}")
print(f"Date range sell-out : {sell_out['sales_date'].min().date()} → {sell_out['sales_date'].max().date()}")
print(f"Date range sell-in  : {sell_in['sales_date'].min().date()} → {sell_in['sales_date'].max().date()}")


---
## 2 · Shape & Basic Descriptive Statistics

In [ ]:
# ── 2.1  Head of each dataset ───────────────────────────────────────────────
print("=== OUTLETS (first 3 rows) ===")
display(outlets.head(3))


In [ ]:
display(skus.head(3))

In [ ]:
display(sell_in.head(3))

In [ ]:
display(sell_out.head(3))

In [ ]:
# ── 2.2  dtypes + nulls for every dataset ───────────────────────────────────
for name, df in [("outlets", outlets), ("skus", skus),
                 ("sell_in", sell_in), ("sell_out", sell_out)]:
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    info = pd.DataFrame({
        "dtype":    df.dtypes,
        "nulls":    df.isnull().sum(),
        "null_%":   null_pct,
        "nunique":  df.nunique(),
    })
    print(f"\n{'='*55}")
    print(f"  {name.upper()}  — {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*55}")
    display(info)


In [ ]:
# ── 2.3  Numeric describe for transactional files ───────────────────────────
print("=== SELL-OUT numeric summary ===")
display(sell_out[["vol_unit","vol_packs"]].describe(percentiles=[.25,.5,.75,.9,.95,.99]))

print("\n=== SELL-IN numeric summary ===")
display(sell_in[["base_unit","sales_unit","lead_days"]].describe(percentiles=[.25,.5,.75,.9,.95,.99]))


In [ ]:
# ── 2.4  Key counts summary table ────────────────────────────────────────────
summary = {
    "Total sell-out records"       : len(sell_out),
    "Total sell-out units (raw)"   : sell_out["vol_unit"].sum(),
    "Total sell-out units (filt.)" : sell_out_f["vol_unit"].sum(),
    "Unique POS (sell-out)"        : sell_out_f["pos"].nunique(),
    "Unique SKUs (sell-out)"       : sell_out_f["sku"].nunique(),
    "Total sell-in records"        : len(sell_in),
    "Total sell-in base units"     : sell_in_f["base_unit"].sum(),
    "Unique outlets (sell-in)"     : sell_in_f["outlet_sf"].nunique(),
    "Active outlets (master)"      : (outlets["status"]=="Active").sum(),
    "COCO outlets"                 : (outlets["ownership_type"]=="COCO").sum(),
    "Independent outlets"          : (outlets["ownership_type"]=="Independent").sum(),
    "Trade SKUs total"             : len(trade_skus),
    "Active trade SKUs"            : (trade_skus["sku_status"]=="Active").sum(),
    "Negative sell-out rows"       : (sell_out["vol_unit"]<0).sum(),
    "Negative sell-in rows"        : (sell_in["base_unit"]<0).sum(),
}
summary_df = pd.DataFrame(list(summary.items()), columns=["Metric","Value"])
summary_df["Value"] = summary_df["Value"].apply(lambda x: f"{x:,.0f}")
display(summary_df.style.set_caption("Key Dataset Metrics"))


---
## 3 · Outlet Master Data Analysis

In [ ]:
# ── 3.1  Status & ownership overview ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Outlet Master Data — Status, Ownership & Territory", fontsize=13)

# Status
status_c = outlets["status"].value_counts()
axes[0].bar(status_c.index, status_c.values, color=[TEAL, AMBER], edgecolor="white")
axes[0].set_title("Outlet Status")
axes[0].set_ylabel("Count")
for bar, v in zip(axes[0].patches, status_c.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+4,
                 str(v), ha="center", fontweight="bold")

# Ownership pie
own_c = outlets["ownership_type"].value_counts()
axes[1].pie(own_c.values, labels=own_c.index, autopct="%1.1f%%",
            colors=[BLUE, PURPLE, GRAY][:len(own_c)], startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=1.5), textprops=dict(fontsize=9))
axes[1].set_title("Ownership Type")

# Outlets per territory (top 15)
terr_c = outlets["terr_name"].value_counts().head(15)
y_pos  = range(len(terr_c))
axes[2].barh(list(y_pos), terr_c.values, color=PURPLE, alpha=0.8)
labels = [str(t).split("SECTEUR_")[-1][:20] for t in terr_c.index]
axes[2].set_yticks(list(y_pos)); axes[2].set_yticklabels(labels, fontsize=7)
axes[2].set_title("Outlets per Territory (top 15)")
axes[2].set_xlabel("Count")

plt.tight_layout(); plt.show()


In [ ]:
# ── 3.2  GPS geographic scatter ──────────────────────────────────────────────
outlets_geo = outlets[(outlets["latitude"] > 48) & (outlets["longitude"] > 1.5)].copy()
own_color   = {"COCO": BLUE, "Independent": CORAL}

fig, ax = plt.subplots(figsize=(9, 7))
for otype, grp in outlets_geo.groupby("ownership_type"):
    ax.scatter(grp["longitude"], grp["latitude"],
               c=own_color.get(otype, GRAY), alpha=0.55, s=25,
               label=otype, edgecolors="none")
ax.set_title("Paris Outlet Geographic Distribution")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.legend(handles=[Patch(color=v, label=k) for k, v in own_color.items()])
plt.tight_layout(); plt.show()

# ── Check for missing / zero GPS ────────────────────────────────────────────
bad_gps = outlets[(outlets["latitude"]==0) | (outlets["longitude"]==0)]
print(f"Outlets with zero/missing GPS: {len(bad_gps)}")
display(bad_gps[["store_code","outlet_name","status","ownership_type"]].head(10))


In [ ]:
# ── 3.3  Territory-level outlet counts ──────────────────────────────────────
print("=== Outlets per Area ===")
display(outlets["area"].value_counts().rename("outlet_count").to_frame())

print("\n=== Ownership breakdown per Area ===")
display(outlets.groupby(["area","ownership_type"]).size()
        .unstack(fill_value=0)
        .sort_values(by=outlets["ownership_type"].value_counts().index[0], ascending=False))


---
## 4 · SKU Catalogue Analysis

In [ ]:
# ── 4.1  Record type, category & status ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("SKU Catalogue — Record Types, Categories & Status")

# Record type
rt = skus["record_type"].value_counts()
axes[0].bar(rt.index, rt.values, color=[BLUE, CORAL], edgecolor="white")
axes[0].set_title("Record Types"); axes[0].set_ylabel("Count")
for bar, v in zip(axes[0].patches, rt.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 str(v), ha="center", fontweight="bold")

# Category (trade SKUs only)
cat = trade_skus["category"].value_counts().dropna()
bars = axes[1].barh(cat.index, cat.values, color=PALETTE[:len(cat)])
axes[1].set_title("Trade SKUs by Category"); axes[1].set_xlabel("Count")
for bar, v in zip(bars, cat.values):
    axes[1].text(bar.get_width()+3, bar.get_y()+bar.get_height()/2,
                 str(v), va="center", fontsize=8)

# Active/Inactive per category
cs = trade_skus.groupby(["category","sku_status"]).size().unstack(fill_value=0)
cs.plot(kind="barh", ax=axes[2], color=[TEAL, RED], stacked=True, edgecolor="white")
axes[2].set_title("Active vs Inactive by Category"); axes[2].set_xlabel("Count")
axes[2].legend(loc="lower right")

plt.tight_layout(); plt.show()


In [ ]:
# ── 4.2  Brand family breakdown ──────────────────────────────────────────────
bf = trade_skus.groupby("brand_family")["sku_name"].count().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(bf.index[::-1], bf.values[::-1], color=TEAL, alpha=0.85)
ax.set_title("Top 20 Brand Families by SKU Count (Trade SKUs)")
ax.set_xlabel("Number of SKUs")
plt.tight_layout(); plt.show()

print("\n=== Full brand-family × category matrix ===")
display(trade_skus.groupby(["brand_family","category"]).size()
        .unstack(fill_value=0)
        .sort_values(by=trade_skus["category"].value_counts().index[0], ascending=False)
        .head(20))


---
## 5 · Sell-Out Temporal Analysis

In [ ]:
# ── 5.1  Daily raw vs filtered ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle("Sell-Out Daily Volume — Raw vs SKU-99999 Filtered")

for ax in axes:
    for d in pd.date_range("2026-01-01","2026-01-31"):
        if d.dayofweek >= 5:
            ax.axvspan(d-pd.Timedelta(hours=12), d+pd.Timedelta(hours=12),
                       alpha=0.1, color=AMBER, zorder=0)

axes[0].bar(daily_raw.index, daily_raw.values, color=BLUE, alpha=0.8, width=0.8, label="Raw")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Raw (includes SKU 99999)"); axes[0].set_ylabel("Units")

axes[1].bar(daily_fil.index, daily_fil.values, color=TEAL, alpha=0.8, width=0.8, label="Filtered")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].set_title("Filtered (excl. SKU 99999)"); axes[1].set_ylabel("Units")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout(); plt.show()
print(f"SKU 99999 share of total: {sell_out[sell_out['sku']==99999]['vol_unit'].sum()/sell_out['vol_unit'].sum()*100:.1f}%")


In [ ]:
# ── 5.2  Day-of-week pattern ─────────────────────────────────────────────────
dow_so = sell_out_f.groupby("dow")["vol_unit"].agg(["sum","mean","std"]).reindex(DOW_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Sell-Out: Day-of-Week Patterns")

colors_dow = [BLUE]*5 + [CORAL, RED]
axes[0].bar(range(7), dow_so["sum"].values, color=colors_dow)
axes[0].set_xticks(range(7))
axes[0].set_xticklabels([d[:3] for d in DOW_ORDER])
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Total Volume by Day of Week"); axes[0].set_ylabel("Total Units")
for i, v in enumerate(dow_so["sum"].values):
    axes[0].text(i, v+20000, fmt_k(v), ha="center", fontsize=8, fontweight="bold")

axes[1].bar(range(7), dow_so["mean"].values, color=colors_dow,
            yerr=dow_so["std"].values, capsize=5,
            error_kw=dict(linewidth=1.2, ecolor="#555555"))
axes[1].set_xticks(range(7))
axes[1].set_xticklabels([d[:3] for d in DOW_ORDER])
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].set_title("Mean ± Std Daily Volume by DoW"); axes[1].set_ylabel("Mean Units")

plt.tight_layout(); plt.show()

print("\n=== Day-of-week summary table ===")
display(dow_so.round(0).astype(int))


In [ ]:
# ── 5.3  Weekly aggregation & week-over-week growth ─────────────────────────
weekly_so = sell_out_f.groupby("week")["vol_unit"].sum()
week_labels = [f"Wk{w}" for w in weekly_so.index]
wow = weekly_so.pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Sell-Out: Weekly Aggregation & Growth")

axes[0].bar(range(len(weekly_so)), weekly_so.values, color=BLUE, alpha=0.85)
axes[0].set_xticks(range(len(weekly_so)))
axes[0].set_xticklabels(week_labels)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Weekly Total Volume"); axes[0].set_ylabel("Units")
for i, v in enumerate(weekly_so.values):
    axes[0].text(i, v+30000, fmt_k(v), ha="center", fontsize=9, fontweight="bold")

bar_colors = [TEAL if v >= 0 else RED for v in wow.values[1:]]
axes[1].bar(range(1, len(wow)), wow.values[1:], color=bar_colors, alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xticks(range(1, len(wow)))
axes[1].set_xticklabels(week_labels[1:])
axes[1].set_title("Week-over-Week Growth (%)"); axes[1].set_ylabel("% Change")
for i, v in enumerate(wow.values[1:]):
    axes[1].text(i+1, v+(1 if v>=0 else -2), f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout(); plt.show()


In [ ]:
# ── 5.4  7-day rolling mean + cumulative volume ──────────────────────────────
rolling7 = daily_fil.rolling(7, min_periods=1).mean()
cumsum   = daily_fil.cumsum()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle("Sell-Out: Rolling Average & Cumulative Volume")

axes[0].bar(daily_fil.index, daily_fil.values, color=BLUE, alpha=0.35, width=0.8, label="Daily")
axes[0].plot(daily_fil.index, rolling7, color=CORAL, linewidth=2.2, label="7-day rolling mean")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Daily Volume + 7-Day Rolling Mean"); axes[0].set_ylabel("Units")
axes[0].legend()

axes[1].plot(cumsum.index, cumsum.values, color=TEAL, linewidth=2.5, marker="o", markersize=3)
axes[1].fill_between(cumsum.index, cumsum.values, alpha=0.15, color=TEAL)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].set_title("Cumulative Sell-Out Volume (Jan 2026)"); axes[1].set_ylabel("Cumulative Units")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout(); plt.show()


In [ ]:
# ── 5.5  Per-transaction volume distribution ─────────────────────────────────
vols_pos = sell_out_f[sell_out_f["vol_unit"] > 0]["vol_unit"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Sell-Out: Per-Transaction Volume Distribution")

axes[0].hist(vols_pos.clip(upper=500), bins=60, color=BLUE, edgecolor="white", alpha=0.85)
axes[0].set_title("Volume per Transaction (clipped at 500)")
axes[0].set_xlabel("Units"); axes[0].set_ylabel("Frequency")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
stats_txt = (f"mean={vols_pos.mean():.0f}  median={vols_pos.median():.0f}\n"
             f"skew={skew(vols_pos):.2f}  kurt={kurtosis(vols_pos):.2f}")
axes[0].text(0.96, 0.95, stats_txt, transform=axes[0].transAxes,
             ha="right", va="top", fontsize=8,
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cccccc"))

axes[1].hist(np.log1p(vols_pos), bins=50, color=PURPLE, edgecolor="white", alpha=0.85)
axes[1].set_title("Log-Transformed Volume (right-skew check)")
axes[1].set_xlabel("log(1 + units)"); axes[1].set_ylabel("Frequency")

plt.tight_layout(); plt.show()


---
## 6 · Sell-Out — SKU & Category Analysis

In [ ]:
# ── 6.1  Category volume breakdown ───────────────────────────────────────────
so_cat    = sell_out_f.groupby("category")["vol_unit"].sum().sort_values(ascending=False)
so_cat_tx = sell_out_f.groupby("category")["vol_unit"].count().reindex(so_cat.index)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Sell-Out by Product Category (excl. SKU 99999)")

bars = axes[0].bar(range(len(so_cat)), so_cat.values,
                   color=PALETTE[:len(so_cat)], edgecolor="white")
axes[0].set_xticks(range(len(so_cat)))
axes[0].set_xticklabels(so_cat.index, rotation=30, ha="right")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Total Volume"); axes[0].set_ylabel("Units")
for bar, v in zip(bars, so_cat.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2000,
                 fmt_k(v), ha="center", fontsize=8, fontweight="bold")

axes[1].pie(so_cat.values, labels=so_cat.index, autopct="%1.1f%%",
            colors=PALETTE[:len(so_cat)], startangle=140,
            wedgeprops=dict(edgecolor="white", linewidth=1.5),
            textprops=dict(fontsize=8))
axes[1].set_title("Volume Share")

axes[2].bar(range(len(so_cat_tx)), so_cat_tx.values,
            color=PALETTE[:len(so_cat_tx)], edgecolor="white")
axes[2].set_xticks(range(len(so_cat_tx)))
axes[2].set_xticklabels(so_cat_tx.index, rotation=30, ha="right")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[2].set_title("Transaction Count"); axes[2].set_ylabel("Transactions")

plt.tight_layout(); plt.show()

print("\n=== Category summary table ===")
cat_summary = pd.DataFrame({
    "total_volume": so_cat,
    "transactions": so_cat_tx,
    "vol_%":        (so_cat / so_cat.sum() * 100).round(2),
    "avg_per_tx":   (so_cat / so_cat_tx).round(1)
})
display(cat_summary)


In [ ]:
# ── 6.2  Top 20 SKUs + Pareto curve ─────────────────────────────────────────
top20_so    = sell_out_f.groupby("sku")["vol_unit"].sum().sort_values(ascending=False).head(20)
sorted_all  = sell_out_f.groupby("sku")["vol_unit"].sum().sort_values(ascending=False)
cum_pct     = sorted_all.cumsum() / sorted_all.sum() * 100
n80         = int((cum_pct >= 80).argmax()) + 1

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Top 20 SKUs + Pareto Curve (excl. SKU 99999)")

axes[0].barh(range(20), top20_so.values[::-1], color=BLUE, alpha=0.85)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels([str(s) for s in top20_so.index[::-1]], fontsize=8)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Top 20 SKUs by Volume"); axes[0].set_xlabel("Units")
for i, v in enumerate(top20_so.values[::-1]):
    axes[0].text(v+500, i, fmt_k(v), va="center", fontsize=7)

axes[1].plot(range(1, len(cum_pct)+1), cum_pct.values, color=CORAL, linewidth=2)
axes[1].axhline(80, color=GRAY, linestyle="--", linewidth=1, label="80% threshold")
axes[1].axvline(n80, color=AMBER, linestyle="--", linewidth=1, label=f"{n80} SKUs → 80%")
axes[1].set_title("Pareto: Cumulative Volume Share")
axes[1].set_xlabel("Number of SKUs (ranked)"); axes[1].set_ylabel("Cumulative %")
axes[1].set_xlim(0, min(80, len(cum_pct)))
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"  → {n80} SKUs account for 80% of total volume ({sell_out_f['sku'].nunique()} total active SKUs)")


In [ ]:
# ── 6.3  Daily trend per top-4 categories ────────────────────────────────────
daily_cat = sell_out_f.groupby(["sales_date","category"])["vol_unit"].sum().unstack(fill_value=0)
top_cats  = so_cat.head(4).index.tolist()

fig, axes = plt.subplots(len(top_cats), 1, figsize=(14, 12), sharex=True)
fig.suptitle("Daily Volume by Top 4 Categories", y=1.01)

for ax, cat, color in zip(axes, top_cats, PALETTE):
    if cat in daily_cat.columns:
        ax.bar(daily_cat.index, daily_cat[cat], color=color, alpha=0.8, width=0.8)
        ax.plot(daily_cat.index, daily_cat[cat].rolling(7, min_periods=1).mean(),
                color="black", linewidth=1.5, linestyle="--", alpha=0.7, label="7d avg")
    ax.set_ylabel("Units"); ax.set_title(cat)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.tight_layout(); plt.show()


In [ ]:
# ── 6.4  Category × Day-of-Week heatmap ─────────────────────────────────────
hm_data = sell_out_f.groupby(["category","dow"])["vol_unit"].sum().unstack(fill_value=0)
hm_data = hm_data.reindex(columns=DOW_ORDER, fill_value=0)
hm_norm = hm_data.div(hm_data.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Category × Day-of-Week Heatmap")

sns.heatmap(hm_data, ax=axes[0], cmap="Blues", annot=True, fmt=".0f",
            linewidths=0.5, cbar_kws={"label":"Units"})
axes[0].set_title("Absolute Volume")
axes[0].set_xticklabels([d[:3] for d in DOW_ORDER], rotation=30)

sns.heatmap(hm_norm, ax=axes[1], cmap="YlOrRd", annot=True, fmt=".1f",
            linewidths=0.5, cbar_kws={"label":"Row %"})
axes[1].set_title("Row-Normalised (%)")
axes[1].set_xticklabels([d[:3] for d in DOW_ORDER], rotation=30)

plt.tight_layout(); plt.show()


---
## 7 · Sell-Out — POS / Outlet Analysis

In [ ]:
# ── 7.1  Volume per POS: top 20, distribution, scatter ──────────────────────
pos_vol = sell_out_f.groupby("pos")["vol_unit"].sum().sort_values(ascending=False)
pos_tx  = sell_out_f.groupby("pos")["vol_unit"].count()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Sell-Out: POS-Level Volume Analysis")

# Top 20
axes[0].barh(range(20), pos_vol.head(20).values[::-1], color=BLUE, alpha=0.85)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels([str(p) for p in pos_vol.head(20).index[::-1]], fontsize=8)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Top 20 POS by Volume"); axes[0].set_xlabel("Units")

# Distribution
q50, q75 = pos_vol.median(), pos_vol.quantile(0.75)
axes[1].hist(pos_vol.values, bins=30, color=CORAL, edgecolor="white", alpha=0.85)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].axvline(q50, color=BLUE, linestyle="--", linewidth=1.5, label=f"Median={fmt_k(q50)}")
axes[1].axvline(q75, color=TEAL, linestyle="--", linewidth=1.5, label=f"Q75={fmt_k(q75)}")
axes[1].set_title("Distribution of POS Total Volumes"); axes[1].legend()

# Scatter: tx count vs volume
corr_r, _ = pearsonr(pos_tx, pos_vol.reindex(pos_tx.index))
axes[2].scatter(pos_tx.values, pos_vol.reindex(pos_tx.index).values,
                alpha=0.55, color=PURPLE, s=30, edgecolors="none")
axes[2].set_title(f"Transactions vs Volume\nPearson r={corr_r:.3f}")
axes[2].set_xlabel("Transactions"); axes[2].set_ylabel("Total Units")
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))

plt.tight_layout(); plt.show()

print("\n=== POS volume percentiles ===")
display(pos_vol.describe(percentiles=[.1,.25,.5,.75,.9,.95]).rename("volume").to_frame())


In [ ]:
# ── 7.2  Top-20 POS daily heatmap ────────────────────────────────────────────
top20_pos  = pos_vol.head(20).index
pos_daily  = (sell_out_f[sell_out_f["pos"].isin(top20_pos)]
              .groupby(["pos","sales_date"])["vol_unit"].sum()
              .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(pos_daily, ax=ax, cmap="YlOrBr",
            xticklabels=pos_daily.columns.strftime("%d"),
            yticklabels=[str(p) for p in pos_daily.index],
            linewidths=0.2, cbar_kws={"label":"Units"})
ax.set_title("Top 20 POS — Daily Volume Heatmap (Jan 2026)")
ax.set_xlabel("Day of January"); ax.set_ylabel("POS Code")
plt.tight_layout(); plt.show()


In [ ]:
# ── 7.3  Zero-sales days distribution ───────────────────────────────────────
all_dates      = pd.date_range("2026-01-01","2026-01-31")
pos_date_full  = (sell_out_f.groupby(["pos","sales_date"])["vol_unit"]
                  .sum().unstack(fill_value=0)
                  .reindex(columns=all_dates, fill_value=0))
zero_days      = (pos_date_full == 0).sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(zero_days.values, bins=31, color=AMBER, edgecolor="white", alpha=0.9)
ax.axvline(zero_days.median(), color=RED, linestyle="--", linewidth=1.5,
           label=f"Median = {zero_days.median():.0f} zero-sales days")
ax.set_title("Distribution of Zero-Sales Days per POS (Jan 2026)")
ax.set_xlabel("Days with zero sales"); ax.set_ylabel("Number of POS")
ax.legend()
plt.tight_layout(); plt.show()

print(f"POS with 0 zero-sales days (active every day): {(zero_days==0).sum()}")
print(f"POS with >20 zero-sales days:                  {(zero_days>20).sum()}")


In [ ]:
# ── 7.4  Volume by ownership type (boxplot) ──────────────────────────────────
so_outlets = (sell_out_f.groupby("outlet_sf")
              .agg(total_vol=("vol_unit","sum"),
                   unique_skus=("sku","nunique"))
              .reset_index()
              .merge(outlets[["outlet_sf","ownership_type"]], on="outlet_sf", how="left"))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sell-Out: Volume & SKU Breadth by Ownership Type")

for ax, col, title in zip(axes,
    ["total_vol","unique_skus"],
    ["Total Volume (Jan)", "Unique SKUs Sold"]):
    groups = [grp[col].values
              for _, grp in so_outlets.dropna(subset=["ownership_type"]).groupby("ownership_type")]
    labels = so_outlets["ownership_type"].dropna().unique().tolist()
    bplot  = ax.boxplot(groups, patch_artist=True,
                        medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bplot["boxes"], [BLUE, PURPLE]):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(labels); ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))

plt.tight_layout(); plt.show()


---
## 8 · Sell-In (Logista) Analysis

In [ ]:
# ── 8.1  Daily sell-in raw vs filtered ──────────────────────────────────────
si_daily_raw = sell_in.groupby("sales_date")["base_unit"].sum()
si_daily_fil = sell_in_f.groupby("sales_date")["base_unit"].sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle("Sell-In: Daily Volume — Raw vs Filtered")

axes[0].bar(si_daily_raw.index, si_daily_raw.values, color=CORAL, alpha=0.8, width=0.6)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Raw (includes SKU 99999)"); axes[0].set_ylabel("Base Units")

axes[1].bar(si_daily_fil.index, si_daily_fil.values, color=AMBER, alpha=0.8, width=0.6)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].set_title("Filtered (excl. SKU 99999)"); axes[1].set_ylabel("Base Units")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.tight_layout(); plt.show()


In [ ]:
# ── 8.2  Day-of-week + delivery lead time ────────────────────────────────────
si_dow  = sell_in_f.groupby("dow")["base_unit"].sum().reindex(DOW_ORDER).dropna()
lead    = sell_in_f["lead_days"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Sell-In: Day-of-Week Pattern & Delivery Lead Times")

axes[0].bar(range(len(si_dow)), si_dow.values, color=CORAL, alpha=0.85)
axes[0].set_xticks(range(len(si_dow)))
axes[0].set_xticklabels([d[:3] for d in si_dow.index])
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Sell-In Volume by Day of Week"); axes[0].set_ylabel("Base Units")

lead_pos = lead[lead >= 0]
axes[1].hist(lead_pos, bins=range(0, int(lead_pos.max())+2), color=PURPLE,
             edgecolor="white", alpha=0.85, align="left")
axes[1].set_title(f"Delivery Lead Time (Sale → Delivery)\n"
                  f"Mean={lead_pos.mean():.1f}d  Median={lead_pos.median():.0f}d")
axes[1].set_xlabel("Days"); axes[1].set_ylabel("Count")

plt.tight_layout(); plt.show()

print("\n=== Lead time summary ===")
display(lead_pos.describe().rename("lead_days").to_frame())


In [ ]:
# ── 8.3  Top 20 SKUs sell-in ─────────────────────────────────────────────────
top20_si = sell_in_f.groupby("sku_code")["base_unit"].sum().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(20), top20_si.values[::-1], color=AMBER, alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels([str(s) for s in top20_si.index[::-1]], fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.set_title("Top 20 SKUs — Sell-In Volume (excl. 99999)")
ax.set_xlabel("Base Units")
for i, v in enumerate(top20_si.values[::-1]):
    ax.text(v+300, i, fmt_k(v), va="center", fontsize=7)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8.4  Outlet-level sell-in distribution ──────────────────────────────────
si_pos_vol = sell_in_f.groupby("outlet_sf")["base_unit"].sum().sort_values(ascending=False)
si_pos_sku = sell_in_f.groupby("outlet_sf")["sku_code"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sell-In: Outlet Distribution")

axes[0].barh(range(20), si_pos_vol.head(20).values[::-1], color=AMBER, alpha=0.85)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels([str(p)[:22] for p in si_pos_vol.head(20).index[::-1]], fontsize=7)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].set_title("Top 20 Outlets by Sell-In Volume"); axes[0].set_xlabel("Base Units")

axes[1].hist(si_pos_sku.values, bins=30, color=TEAL, edgecolor="white", alpha=0.85)
axes[1].set_title("SKU Breadth per Outlet (Sell-In)")
axes[1].set_xlabel("Unique SKUs"); axes[1].set_ylabel("Number of Outlets")

plt.tight_layout(); plt.show()


---
## 9 · Sell-In vs Sell-Out Comparison

In [ ]:
# ── 9.1  Normalised daily overlay ────────────────────────────────────────────
all_dr   = pd.date_range("2026-01-01","2026-01-31")
si_norm  = (si_daily_fil / si_daily_fil.max()).reindex(all_dr)
so_norm  = (daily_fil    / daily_fil.max()).reindex(all_dr, fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(all_dr, so_norm.values, color=BLUE, linewidth=2, label="Sell-Out (normalised)")
ax.plot(all_dr, si_norm.values, color=CORAL, linewidth=2, linestyle="--",
        marker="x", markersize=5, label="Sell-In (normalised)")
ax.fill_between(all_dr, so_norm.values, alpha=0.1, color=BLUE)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax.set_title("Sell-In vs Sell-Out — Daily Normalised Volume (Jan 2026)")
ax.set_ylabel("Normalised Volume (0–1)"); ax.legend()
plt.tight_layout(); plt.show()

print("Note: sell-in is weekday-only (batch deliveries); sell-out runs all 31 days.")


In [ ]:
# ── 9.2  Per-SKU scatter + sell-out/sell-in ratio ────────────────────────────
so_sku_vol  = sell_out_f.groupby("sku")["vol_unit"].sum()
si_sku_vol  = sell_in_f.groupby("sku_code")["base_unit"].sum()
shared_skus = set(so_sku_vol.index) & set(si_sku_vol.index)
shared_df   = pd.DataFrame({
    "sell_out": so_sku_vol.reindex(list(shared_skus)),
    "sell_in":  si_sku_vol.reindex(list(shared_skus))
}).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Shared SKU Volume: Sell-In vs Sell-Out (Jan)")

max_val  = max(shared_df.max().max(), 1)
corr_r,_ = pearsonr(shared_df["sell_in"], shared_df["sell_out"])
axes[0].scatter(shared_df["sell_in"], shared_df["sell_out"],
                alpha=0.5, color=PURPLE, s=30, edgecolors="none")
axes[0].plot([0, max_val], [0, max_val], color=GRAY, linestyle="--",
             linewidth=1, label="1:1 line")
axes[0].set_title(f"Sell-In vs Sell-Out per SKU\nPearson r={corr_r:.3f}")
axes[0].set_xlabel("Sell-In Volume"); axes[0].set_ylabel("Sell-Out Volume")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].legend()

ratio = (shared_df["sell_out"]/shared_df["sell_in"]).replace([np.inf,-np.inf], np.nan).dropna().clip(0,5)
axes[1].hist(ratio, bins=40, color=TEAL, edgecolor="white", alpha=0.85)
axes[1].axvline(1.0, color=RED, linestyle="--", linewidth=1.5, label="SI=SO ratio")
axes[1].axvline(ratio.median(), color=BLUE, linestyle="--", linewidth=1.5,
                label=f"Median={ratio.median():.2f}")
axes[1].set_title("Sell-Out / Sell-In Ratio per SKU (clipped at 5)")
axes[1].set_xlabel("Ratio"); axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"Shared SKUs: {len(shared_df)}")


In [ ]:
# ── 9.3  Outlet channel coverage overlap ─────────────────────────────────────
si_set  = set(sell_in_f["outlet_sf"])
so_set  = set(sell_out_f["outlet_sf"])
both    = si_set & so_set
si_only = si_set - so_set
so_only = so_set - si_set

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    ["Sell-In only\n(Logista)","Both channels","Sell-Out only\n(Bimedia)"],
    [len(si_only), len(both), len(so_only)],
    color=[CORAL, TEAL, BLUE], edgecolor="white"
)
for bar, v in zip(bars, [len(si_only), len(both), len(so_only)]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            str(v), ha="center", fontsize=12, fontweight="bold")
ax.set_title("Outlet Coverage: Channel Overlap"); ax.set_ylabel("Number of Outlets")
plt.tight_layout(); plt.show()

print(f"Bimedia covers {len(so_set)}/{len(si_set)} Logista outlets ({len(so_set)/len(si_set)*100:.1f}%)")


---
## 10 · Statistical Analysis

In [ ]:
# ── 10.1  Normality tests on daily sell-out ──────────────────────────────────
from scipy.stats import probplot, shapiro, kstest

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Daily Sell-Out Distribution — Normality Tests")

# Q-Q plot
(osm, osr), (slope, intercept, r) = probplot(daily_fil.values)
axes[0].scatter(osm, osr, s=20, color=BLUE, alpha=0.8, edgecolors="none")
lx = np.array([osm.min(), osm.max()])
axes[0].plot(lx, slope*lx+intercept, color=RED, linewidth=1.5)
axes[0].set_title(f"Q-Q Plot  (r={r:.3f})")
axes[0].set_xlabel("Theoretical Quantiles"); axes[0].set_ylabel("Sample Quantiles")

# Test p-values
stat_sw, p_sw = shapiro(daily_fil.values)
stat_ks, p_ks = kstest(
    (daily_fil.values - daily_fil.mean()) / daily_fil.std(), "norm"
)
axes[1].bar(["Shapiro-Wilk","Kolmogorov-Smirnov"], [p_sw, p_ks],
            color=[TEAL if p > 0.05 else RED for p in [p_sw, p_ks]])
axes[1].axhline(0.05, color=AMBER, linestyle="--", label="α=0.05")
axes[1].set_title("Normality Test p-values"); axes[1].set_ylabel("p-value")
axes[1].legend()
for i, (name, p) in enumerate(zip(["SW","KS"],[p_sw, p_ks])):
    axes[1].text(i, p+0.003, f"p={p:.4f}", ha="center", fontsize=8)

# Histogram + normal fit
x = np.linspace(daily_fil.min(), daily_fil.max(), 200)
mu, sigma = daily_fil.mean(), daily_fil.std()
axes[2].hist(daily_fil.values, bins=12, density=True, color=BLUE, alpha=0.5, edgecolor="white")
axes[2].plot(x, stats.norm.pdf(x, mu, sigma), color=RED, linewidth=2,
             label=f"Normal fit\nμ={fmt_k(mu)} σ={fmt_k(sigma)}")
axes[2].set_title("Histogram + Normal Fit")
axes[2].set_xlabel("Daily Units")
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f"Shapiro-Wilk:        stat={stat_sw:.4f}, p={p_sw:.6f}  → {'NORMAL' if p_sw>0.05 else 'NOT normal'}")
print(f"Kolmogorov-Smirnov:  stat={stat_ks:.4f}, p={p_ks:.6f}  → {'NORMAL' if p_ks>0.05 else 'NOT normal'}")


In [ ]:
# ── 10.2  Outlier detection on per-transaction volumes ───────────────────────
vol_data  = sell_out_f[sell_out_f["vol_unit"] > 0]["vol_unit"].copy()
z_scores  = np.abs(stats.zscore(vol_data))
iqr       = vol_data.quantile(0.75) - vol_data.quantile(0.25)
iqr_hi    = vol_data.quantile(0.75) + 1.5*iqr
n_z       = (z_scores > 3).sum()
n_iqr     = (vol_data > iqr_hi).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Outlier Detection — Sell-Out Per-Transaction Volume")

axes[0].boxplot(vol_data.clip(upper=2000), vert=False, patch_artist=True,
                boxprops=dict(facecolor=BLUE, alpha=0.6),
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker=".", color=RED, alpha=0.3, markersize=3))
axes[0].set_title(f"Boxplot (display clipped at 2000)\nIQR outliers: {n_iqr:,}   Z>3: {n_z:,}")
axes[0].set_xlabel("Volume per Transaction")

vol_sorted = vol_data.sort_values().reset_index(drop=True)
axes[1].plot(vol_sorted.index, vol_sorted.values, color=PURPLE, linewidth=1.2)
axes[1].axhline(iqr_hi, color=RED, linestyle="--", linewidth=1.2,
                label=f"IQR upper fence = {iqr_hi:.0f}")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].set_title("Sorted Volume (Rank Order)")
axes[1].set_xlabel("Rank"); axes[1].set_ylabel("Volume")
axes[1].legend()

plt.tight_layout(); plt.show()
print(f"IQR upper fence : {iqr_hi:.0f}")
print(f"Z-score>3 outliers: {n_z:,}  ({n_z/len(vol_data)*100:.2f}%)")
print(f"IQR outliers:       {n_iqr:,}  ({n_iqr/len(vol_data)*100:.2f}%)")


In [ ]:
# ── 10.3  Correlation matrix — POS-level features ────────────────────────────
pos_features = sell_out_f.groupby("pos").agg(
    total_vol   = ("vol_unit","sum"),
    mean_vol    = ("vol_unit","mean"),
    tx_count    = ("vol_unit","count"),
    unique_skus = ("sku","nunique"),
    active_days = ("sales_date","nunique"),
    std_vol     = ("vol_unit","std"),
).fillna(0)

corr_matrix = pos_features.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=ax, cmap="coolwarm", annot=True, fmt=".2f",
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={"label":"Pearson r"})
ax.set_title("Correlation Matrix — POS-Level Features")
plt.tight_layout(); plt.show()


In [ ]:
# ── 10.4  Autocorrelation Function (ACF) of daily sell-out ───────────────────
series   = daily_fil.values - daily_fil.mean()
acf_full = np.correlate(series, series, mode="full")
acf_norm = acf_full[len(acf_full)//2:] / acf_full[len(acf_full)//2:][0]
n        = len(series)
conf     = 1.96 / np.sqrt(n)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(16), acf_norm[:16], color=BLUE, alpha=0.8, width=0.6)
ax.axhline( conf, color=RED, linestyle="--", linewidth=1, label=f"+95% CI ({conf:.2f})")
ax.axhline(-conf, color=RED, linestyle="--", linewidth=1, label=f"−95% CI")
ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("ACF — Daily Sell-Out (excl. SKU 99999)")
ax.set_xlabel("Lag (days)"); ax.set_ylabel("ACF")
ax.set_xticks(range(16)); ax.legend()
plt.tight_layout(); plt.show()

sig_lags = [i for i in range(1,16) if abs(acf_norm[i]) > conf]
print(f"Statistically significant lags (|ACF| > {conf:.2f}): {sig_lags}")


---
## 11 · Data Quality & Returns Analysis

In [ ]:
# ── 11.1  Negative / return transactions overview ────────────────────────────
so_neg = sell_out[sell_out["vol_unit"] < 0].copy()
si_neg = sell_in[sell_in["base_unit"] < 0].copy()

print(f"Sell-out returns: {len(so_neg):,} rows  |  {so_neg['vol_unit'].sum():,} units")
print(f"Sell-in returns:  {len(si_neg):,} rows  |  {si_neg['base_unit'].sum():,} base units")
print(f"Return rate (rows): sell-out={len(so_neg)/len(sell_out)*100:.2f}%  "
      f"sell-in={len(si_neg)/len(sell_in)*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Returns & Negative Transactions")

neg_daily = so_neg.groupby("sales_date")["vol_unit"].sum()
axes[0].bar(neg_daily.index, np.abs(neg_daily.values), color=RED, alpha=0.8, width=0.8)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d"))
axes[0].set_title("Sell-Out Returns — Daily (abs. units)")
axes[0].set_xlabel("Day (Jan)"); axes[0].set_ylabel("Abs. Units Returned")

si_neg_daily = si_neg.groupby("sales_date")["base_unit"].sum()
axes[1].bar(si_neg_daily.index, np.abs(si_neg_daily.values), color=AMBER, alpha=0.8, width=0.6)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d"))
axes[1].set_title("Sell-In Returns — Daily (abs. units)")
axes[1].set_xlabel("Day (Jan)"); axes[1].set_ylabel("Abs. Base Units")

plt.tight_layout(); plt.show()


In [ ]:
# ── 11.2  Missing values heatmap ─────────────────────────────────────────────
frames   = {"outlets": outlets, "skus": skus, "sell_in": sell_in, "sell_out": sell_out}
miss_df  = pd.DataFrame({
    name: df.isnull().sum() / len(df) * 100 for name, df in frames.items()
}).fillna(0)
miss_df  = miss_df[(miss_df > 0).any(axis=1)]

if len(miss_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss_df)*0.45)))
    sns.heatmap(miss_df, ax=ax, cmap="Oranges", annot=True, fmt=".1f",
                linewidths=0.5, cbar_kws={"label":"% Missing"})
    ax.set_title("Missing Value % by Dataset & Column")
    plt.tight_layout(); plt.show()
else:
    print("No missing values found across all four datasets.")


In [ ]:
# ── 11.3  Duplicate check ────────────────────────────────────────────────────
for name, df, key_cols in [
    ("sell_out", sell_out, ["sales_date","pos","sku"]),
    ("sell_in",  sell_in,  ["sales_date","pos_code","sku_code"]),
    ("outlets",  outlets,  ["outlet_sf"]),
    ("skus",     skus,     ["sku_code"]),
]:
    dups = df.duplicated(subset=key_cols).sum()
    print(f"{name:12s}  key={key_cols}  →  duplicates: {dups:,}")


---
## 12 · Advanced — SKU Velocity Segmentation (A/B/C)

In [ ]:
# ── 12.1  Build SKU metrics ──────────────────────────────────────────────────
sku_metrics = sell_out_f.groupby("sku").agg(
    total_vol    = ("vol_unit","sum"),
    active_days  = ("sales_date","nunique"),
    store_count  = ("pos","nunique"),
    tx_count     = ("vol_unit","count"),
).reset_index()

sku_metrics["avg_daily_vel"]   = sku_metrics["total_vol"] / sku_metrics["active_days"]
sku_metrics["avg_vol_per_store"]= sku_metrics["total_vol"] / sku_metrics["store_count"]
sku_metrics                    = sku_metrics.sort_values("total_vol", ascending=False).reset_index(drop=True)
sku_metrics["cum_pct"]         = sku_metrics["total_vol"].cumsum() / sku_metrics["total_vol"].sum() * 100

sku_metrics["segment"] = "C"
sku_metrics.loc[sku_metrics["cum_pct"] <= 80, "segment"] = "A"
sku_metrics.loc[(sku_metrics["cum_pct"] > 80) & (sku_metrics["cum_pct"] <= 95), "segment"] = "B"

seg_summary = sku_metrics.groupby("segment").agg(
    sku_count  = ("sku","count"),
    total_vol  = ("total_vol","sum"),
).assign(vol_pct=lambda x: x["total_vol"]/sku_metrics["total_vol"].sum()*100)

print("=== A/B/C Segment Summary ===")
display(seg_summary.round(2))


In [ ]:
# ── 12.2  Visualise segmentation ─────────────────────────────────────────────
seg_colors = {"A": BLUE, "B": AMBER, "C": CORAL}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("SKU Velocity Segmentation (A/B/C)")

sc_c = [seg_colors[s] for s in sku_metrics["segment"]]
axes[0].scatter(sku_metrics["store_count"], sku_metrics["avg_daily_vel"],
                c=sc_c, alpha=0.6, s=25, edgecolors="none")
axes[0].set_xlabel("Store Count"); axes[0].set_ylabel("Avg Daily Velocity")
axes[0].set_title("Velocity vs Distribution")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].legend(handles=[Patch(color=v, label=k) for k, v in seg_colors.items()], title="Segment")

axes[1].bar(seg_summary.index, seg_summary["vol_pct"],
            color=[seg_colors[s] for s in seg_summary.index], edgecolor="white")
axes[1].set_title("Volume Share by Segment"); axes[1].set_ylabel("% of Total Volume")
for i, (s, row) in enumerate(seg_summary.iterrows()):
    axes[1].text(i, row["vol_pct"]+0.5, f"{row['vol_pct']:.1f}%\n({row['sku_count']} SKUs)",
                 ha="center", fontsize=9)

axes[2].bar(seg_summary.index, seg_summary["sku_count"],
            color=[seg_colors[s] for s in seg_summary.index], edgecolor="white")
axes[2].set_title("SKU Count by Segment"); axes[2].set_ylabel("Number of SKUs")
for i, (s, row) in enumerate(seg_summary.iterrows()):
    axes[2].text(i, row["sku_count"]+0.3, str(row["sku_count"]), ha="center", fontweight="bold")

plt.tight_layout(); plt.show()


In [ ]:
# ── 12.3  SKU penetration rate ───────────────────────────────────────────────
total_pos       = sell_out_f["pos"].nunique()
sku_penetration = sell_out_f.groupby("sku")["pos"].nunique().sort_values(ascending=False).head(30)
pen_rate        = sku_penetration / total_pos * 100

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = [TEAL if v >= 50 else (AMBER if v >= 25 else CORAL) for v in pen_rate.values]
ax.bar(range(len(pen_rate)), pen_rate.values, color=bar_colors, edgecolor="white", alpha=0.85)
ax.set_xticks(range(len(pen_rate)))
ax.set_xticklabels([str(s) for s in pen_rate.index], rotation=45, ha="right", fontsize=8)
ax.axhline(50, color=RED, linestyle="--", linewidth=1.2, label="50% stores")
ax.axhline(25, color=AMBER, linestyle="--", linewidth=1.2, label="25% stores")
ax.set_title(f"Top 30 SKU Penetration Rate (% of {total_pos} POS)")
ax.set_ylabel("Store Penetration (%)"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── 12.4  SKU consistency — % of days active ────────────────────────────────
sku_days  = (sell_out_f[sell_out_f["vol_unit"] > 0]
             .groupby("sku")["sales_date"].nunique()
             .sort_values(ascending=False))
sku_cons  = (sku_days / 31 * 100).head(30)

fig, ax = plt.subplots(figsize=(12, 5))
bcolors = [TEAL if v >= 80 else (AMBER if v >= 50 else CORAL) for v in sku_cons.values]
ax.bar(range(len(sku_cons)), sku_cons.values, color=bcolors, edgecolor="white", alpha=0.85)
ax.set_xticks(range(len(sku_cons)))
ax.set_xticklabels([str(s) for s in sku_cons.index], rotation=45, ha="right", fontsize=8)
ax.axhline(80, color=TEAL,  linestyle="--", linewidth=1.2, label="80% of days")
ax.axhline(50, color=AMBER, linestyle="--", linewidth=1.2, label="50% of days")
ax.set_title("Top 30 SKUs — % of Days with Positive Sales")
ax.set_ylabel("% of Days Active"); ax.legend()
plt.tight_layout(); plt.show()

print(f"SKUs active on ALL 31 days: {(sku_days==31).sum()}")
print(f"SKUs active on <5 days:     {(sku_days<5).sum()}")


---
## 13 · Advanced — Store Performance Matrix & Quadrant Classification

In [ ]:
# ── 13.1  Build POS metrics ──────────────────────────────────────────────────
pos_stats = sell_out_f.groupby("pos").agg(
    total_vol    = ("vol_unit","sum"),
    sku_breadth  = ("sku","nunique"),
    active_days  = ("sales_date","nunique"),
    tx_count     = ("vol_unit","count"),
    mean_tx_size = ("vol_unit","mean"),
).reset_index()

print("=== POS-level metrics (head) ===")
display(pos_stats.sort_values("total_vol", ascending=False).head(10).round(1))


In [ ]:
# ── 13.2  Scatter: volume vs SKU breadth (colour = active days) ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Store Performance Matrix")

sc  = axes[0].scatter(pos_stats["sku_breadth"], pos_stats["total_vol"],
                      c=pos_stats["active_days"], cmap="viridis",
                      alpha=0.7, s=40, edgecolors="none")
plt.colorbar(sc, ax=axes[0], label="Active Days")
axes[0].set_xlabel("SKU Breadth"); axes[0].set_ylabel("Total Volume")
axes[0].set_title("Volume vs SKU Breadth (colour = active days)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))

sc2 = axes[1].scatter(pos_stats["tx_count"], pos_stats["total_vol"],
                      c=pos_stats["sku_breadth"], cmap="plasma",
                      alpha=0.7, s=40, edgecolors="none")
plt.colorbar(sc2, ax=axes[1], label="SKU Breadth")
axes[1].set_xlabel("Transaction Count"); axes[1].set_ylabel("Total Volume")
axes[1].set_title("Volume vs Transactions (colour = SKU breadth)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))

plt.tight_layout(); plt.show()


In [ ]:
# ── 13.3  4-quadrant classification ─────────────────────────────────────────
med_vol  = pos_stats["total_vol"].median()
med_skus = pos_stats["sku_breadth"].median()
print(f"Median total volume  : {fmt_k(med_vol)}")
print(f"Median SKU breadth   : {med_skus:.0f}")

quad_colors = {
    "High Vol + Wide SKU":   BLUE,
    "High Vol + Narrow SKU": TEAL,
    "Low Vol + Wide SKU":    AMBER,
    "Low Vol + Narrow SKU":  CORAL,
}
conditions  = [
    (pos_stats["total_vol"] >= med_vol) & (pos_stats["sku_breadth"] >= med_skus),
    (pos_stats["total_vol"] >= med_vol) & (pos_stats["sku_breadth"] <  med_skus),
    (pos_stats["total_vol"] <  med_vol) & (pos_stats["sku_breadth"] >= med_skus),
    (pos_stats["total_vol"] <  med_vol) & (pos_stats["sku_breadth"] <  med_skus),
]
pos_stats["quadrant"] = np.select(conditions, list(quad_colors.keys()), default="Other")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Store 4-Quadrant Classification")

for q, color in quad_colors.items():
    sub = pos_stats[pos_stats["quadrant"]==q]
    axes[0].scatter(sub["sku_breadth"], sub["total_vol"],
                    color=color, label=f"{q} ({len(sub)})", alpha=0.7, s=35)
axes[0].axhline(med_vol,  color=GRAY, linestyle="--", linewidth=0.8)
axes[0].axvline(med_skus, color=GRAY, linestyle="--", linewidth=0.8)
axes[0].set_xlabel("SKU Breadth"); axes[0].set_ylabel("Total Volume")
axes[0].set_title("Quadrant Scatter")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
axes[0].legend(fontsize=7, loc="upper left")

quad_counts = pos_stats["quadrant"].value_counts()
axes[1].bar(range(len(quad_counts)), quad_counts.values,
            color=[quad_colors.get(q, GRAY) for q in quad_counts.index], edgecolor="white")
axes[1].set_xticks(range(len(quad_counts)))
axes[1].set_xticklabels(quad_counts.index, rotation=20, ha="right", fontsize=8)
axes[1].set_title("Store Count per Quadrant"); axes[1].set_ylabel("Count")
for i, v in enumerate(quad_counts.values):
    axes[1].text(i, v+0.3, str(v), ha="center", fontweight="bold")

plt.tight_layout(); plt.show()


---
## 14 · Export Results to CSV

In [ ]:
# ── 14.1  Summary statistics ─────────────────────────────────────────────────
summary_rows = {
    "Total sell-out records"       : len(sell_out),
    "Total sell-out units (raw)"   : sell_out["vol_unit"].sum(),
    "Total sell-out units (filt.)" : sell_out_f["vol_unit"].sum(),
    "Unique POS"                   : sell_out_f["pos"].nunique(),
    "Unique SKUs"                  : sell_out_f["sku"].nunique(),
    "Total sell-in records"        : len(sell_in),
    "Total sell-in base units"     : sell_in_f["base_unit"].sum(),
    "Active outlets (master)"      : (outlets["status"]=="Active").sum(),
    "Negative sell-out rows"       : (sell_out["vol_unit"]<0).sum(),
    "Negative sell-in rows"        : (sell_in["base_unit"]<0).sum(),
}
pd.DataFrame(list(summary_rows.items()), columns=["metric","value"]).to_csv(
    "00_summary_statistics.csv", index=False)

# ── 14.2  SKU velocity segments ──────────────────────────────────────────────
sku_metrics.to_csv("09_sku_velocity_segments.csv", index=False)

# ── 14.3  POS quadrant classification ────────────────────────────────────────
pos_stats.to_csv("10_pos_quadrant_classification.csv", index=False)

print("✅  Exported:")
print("   00_summary_statistics.csv")
print("   09_sku_velocity_segments.csv")
print("   10_pos_quadrant_classification.csv")
